In [101]:
import numpy as np
import nussl
from scipy.io import wavfile
from scipy.ndimage import gaussian_filter
import os
from tqdm import tqdm
from moviepy.video.io.ffmpeg_tools import ffmpeg_extract_audio
import librosa
import time
from pathlib import Path
from IPython.display import Audio
from voicolate.transcribe import transcribe
#from voicolate.isolate import isolate_audio
import joblib

In [102]:
letter_assignments = ['A', 'B', 'C', 'D']

In [103]:
def extract_audio(posix_video_path):
    print("Extracting audio from video files...\n")
    out_audio_path = str(os.path.basename(str(posix_video_path)[:-4]+'.wav'))
    ffmpeg_extract_audio(str(posix_video_path), out_audio_path)
    # librosa returns the numpy array as well as the sample rate
    loaded = librosa.load(out_audio_path, sr=None)
    # delete the file
    os.remove(out_audio_path)
    # returns both the audio as a numpy array and the sample rate (arr, sr)
    return loaded

In [104]:
def arr_to_batch(array, batch_size):
    shape = array.shape[1]
    n_batches = shape // batch_size
    leftover = shape - batch_size*n_batches
    batches = []
    b = 0
    for batch in range(n_batches):
        batches.append(array[:,b:b+batch_size])
        b+=batch_size
    if leftover:
        batches.append(array[:,b:b+leftover])
    return batches

In [105]:
def additive_mix(audio_iter):
    """
    Takes an iterable of audio files and returns their sum.
    """
    # get shapes of all audio files
    shapes = [audio.shape for audio in audio_iter]
    # get the maximum shape
    max_shape = max(shapes)[0]
    # print a warning if any shape is less than 99% of the max shape
    for shape in shapes:
        if shape[0] < max_shape*.99:
            print("Warning: audio file is less than 99% of the max shape.")
    # pad all audio files to the max shape
    audio_iter = [np.pad(arr, (0, max_shape-len(arr)), 'constant') for arr in audio_iter]

    mix = np.zeros(len(audio_iter[0]))
    for audio in audio_iter:
        mix += audio
    return mix


In [106]:
def apply_wiener(file_list, iterations=10, save_to_file=False, output_path=None, return_outputs=True,
                 batch_size=441000, video_file_list=None):
    """
    Takes list of .wav files and returns filtered audio.
    Assumes all audio files are mono and of the *exact* same length.
    If video file list is provided, the audio from the video files will be used as the reference signal (i.e., the "estimates").
    """
    if video_file_list:
        video_audio_signals = [nussl.AudioSignal(extract_audio(i)[0], sample_rate=extract_audio(i)[1]) for i in video_file_list]
        # check if the video audio signals are the same length
        if not all([video_audio_signals[i].audio_data.shape for i in range(len(video_audio_signals))]):
            raise Exception("Video audio files are of different lengths!")
        # check if the video audio signals are the same sample rate. If not, resample.
        if not all([video_audio_signals[i].sample_rate == video_audio_signals[0].sample_rate for i in range(len(video_audio_signals))]):
            print("Resampling video audio files to match sample rate of first video audio file...")
            time.sleep(3)
            for i in range(len(video_audio_signals)):
                video_audio_signals[i].resample(video_audio_signals[0].sample_rate)
        # check if the video audio signals are mono. If not, convert to mono.
        if not all([video_audio_signals[i].num_channels == 1 for i in range(len(video_audio_signals))]):
            print("Converting video audio files to mono...")
            time.sleep(3)
            for i in range(len(video_audio_signals)):
                video_audio_signals[i].convert_channels(1)
        # check if the video audio signals are the same length as the audio files. If not, trim.
        if not all([video_audio_signals[i].audio_data.shape == nussl.AudioSignal(file_list[i]).audio_data.shape for i in range(len(video_audio_signals))]):
            print("Trimming video audio files to match length of audio files...")
            time.sleep(3)
            for i in range(len(video_audio_signals)):
                video_audio_signals[i].trim(0, nussl.AudioSignal(file_list[i]).audio_data.shape[0])
        video_mix = additive_mix([video_audio_signals[i].audio_data[0] for i in range(len(video_audio_signals))])
    
    estimates = [nussl.AudioSignal(i) for i in file_list]
    naud = len(file_list)
    rate = estimates[0].sample_rate
    shape = estimates[0].audio_data.shape[0]
    if not all([estimates[i].audio_data.shape for i in range(naud)]):
        raise Exception("Audio files are of different lengths!")
    batches = arr_to_batch(np.array([estimates[i].audio_data[0] for i in range(naud)]), batch_size=batch_size)
    if video_file_list:
        vid_batches = arr_to_batch(np.array([video_mix]), batch_size=batch_size)
    outs = []
    print("Applying Wiener...\n")
    for b, batch in tqdm(enumerate(batches)):
        mix = additive_mix(batch)
        mix = nussl.core.AudioSignal(audio_data_array=mix, sample_rate=rate)
        if video_file_list:
            # NOTE: This is where the video audio is used as the reference signal.
            #       I may need to try to max each audio file to its detected power in the video mix. 
            wiener = nussl.separation.benchmark.WienerFilter(vid_batches[b],
                                                             [nussl.core.AudioSignal(audio_data_array=i, sample_rate=rate)
                                                              for i in batch], iterations=iterations)
        else:
            wiener = nussl.separation.benchmark.WienerFilter(mix,
                                                            [nussl.core.AudioSignal(audio_data_array=i, sample_rate=rate)
                                                            for i in batch], iterations=iterations)
        wout = wiener()
        outs.append(np.array([i.audio_data[0] for i in wout]))
    outs = np.concatenate(outs, axis=1)
    if save_to_file:
        if output_path == None:
            output_path = os.path.getcwd()
        for f, file in enumerate(file_list):
            out = os.path.join(output_path, os.path.basename(file)[:-4] + '_wiener.wav')
            wavfile.write(out, estimates[0].sample_rate, outs[f])
    if return_outputs:
        return outs

In [107]:
def window_rms(a, rate=44100, window_ms=10):
    """
    Takes a numpy array representing audio and returns rolling-window root-mean-squared value.
    """
    window_size = int(round((rate/1000)*window_ms))
    a2 = np.power(a,2)
    window = np.ones(window_size)/float(window_size)
    return np.sqrt(np.convolve(a2, window, 'same'))

In [108]:
def mask(input_arr, threshold=0.001, sigma=20, rate=44100, window_ms=10, return_mask=False):
    """
    :param input_arr: Array representing audio.
    :type input_arr: np.1darray
    :param threshold: RMS value below which values will be zero'd.
    :param sigma: Sigma value for gaussian filter (higher values = more fade)
    :param rate: Hz of the input audio
    :param window_ms: Window width in ms for getting rms values
    :param return_mask: Whether or not to return the mask used to silence non-partipant speech.
    :type return_mask: bool
    :return: Masked audio (or audio along with mask if return_mask=True
    """
    loud = window_rms(input_arr, rate=rate, window_ms=window_ms)
    # smooth in and outs to reduce choppiness
    loud[np.where(loud != 1)] = gaussian_filter(loud, sigma)[np.where(loud != 1)]
    loud[np.where(loud < threshold)] = 0
    loud[np.where(loud > threshold)] = 1
    clean = loud*input_arr
    if not return_mask:
        return clean
    else:
        return clean, loud

In [109]:
def mask_audio(wiener_outputs, raw_audio, rate=44100, window_ms=10, stride_ms=2, threshold=0.001, sigma=20):
    """
    Gets RMS of Wiener-filtered audio and uses it to mask the original audio to retain quality.
    A gaussian filter is used to smooth in and out phases of speech to reduce choppiness.
    """
    cleaned_outputs = []
    for w, wout in tqdm(enumerate(wiener_outputs)):
        loud = window_rms(wout)
        #smooth in and outs to reduce choppiness
        loud[np.where(loud!=1)] = gaussian_filter(loud, sigma)[np.where(loud!=1)]
        loud[np.where(loud<threshold)] = 0
        loud[np.where(loud>threshold)] = 1
        clean = loud*raw_audio[w]
        cleaned_outputs.append(clean)
    return cleaned_outputs

In [110]:
def save_isolated_audio(array_list, rate=44100, output_path = None, output_name=None):
    if not output_name:
        outnames = [str(i)+'_isolated.wav' for i in range(len(array_list))]
    else:
        outnames = [output_name+'_'+str(i)+'.wav' for i in range(len(array_list))]
    if not output_path:
        output_path = os.getcwd()
    filenames = [os.path.join(output_path, outnames[i]) for i in range(len(array_list))]
    for f, array in enumerate(array_list):
        wavfile.write(filenames[f], rate, array)
    return filenames

In [111]:
# get audio files of all the video files given data dir
def get_video_audio_files(data_dir):
    # assumes files are already trimmed and concatenated
    data_dir = Path(data_dir)
    derivative_path = data_dir / 'derivatives'
    video_files = [str(i) for i in derivative_path.iterdir() if Path(i).suffix == '.mp4']
    video_files = [i for i in video_files if 'concatenated_trimmed' in i]
    video_audio_files = [extract_audio(i) for i in video_files]
    # check if all the same sample rate
    if len(set([i[1] for i in video_audio_files])) > 1 or any([i[1] != 44100 for i in video_audio_files]):
        print(f'Video audio files for {data_dir} are not all the same sample rate. Resampling...')
        # resample
        video_audio_files = [librosa.resample(i[0], i[1], 44100) for i in video_audio_files if i[1] != 44100]
    return video_audio_files

In [112]:
def isolate_audio(file_list, rate=44100, mask_threshold=0.001, sigma=20, save_files=False, output_path=None, video_file_list=None):
    """
    Uses RMS values from Wiener-filtered audio to remove interference. Input is a list of audio files
    Returns numpy vectors representing the cleaned sound.
    """
    print('Applying Wiener Filter, may take a while...\n')
    if video_file_list:
        wiener_outputs = apply_wiener(file_list, video_file_list=video_file_list)
    else:
        wiener_outputs = apply_wiener(file_list)
    raw_audio = [nussl.AudioSignal(f).audio_data[0] for f in file_list]
    print('Masking...\n')
    masked_audio = mask_audio(wiener_outputs, raw_audio, threshold=mask_threshold, sigma=sigma, rate=rate)
    if save_files:
        # check if output path exists. If not, make it.
        if not os.path.exists(output_path):
            os.makedirs(output_path)
        saved_files = save_isolated_audio(masked_audio, rate, output_path)
        return saved_files
    else:
        return masked_audio


In [113]:
def extract_transcripts(data_dir, use_video_audio=True):
    data_dir = Path(data_dir)
    derivative_path = data_dir / 'derivatives'
    audio_files = [str(i) for i in derivative_path.iterdir() if Path(i).suffix == '.wav']

    # check if audio files are all the same length
    audio_durations = []
    for file in audio_files:
        sr, audio = wavfile.read(file)
        audio_durations.append(len(audio)/sr)
    if len(set(audio_durations)) > 1:
        raise ValueError(f'Audio files for {data_dir} are not all the same length.')
    
    if use_video_audio:
        video_files = [str(i) for i in derivative_path.iterdir() if Path(i).suffix == '.mp4']
        video_files = [i for i in video_files if 'concatenated_trimmed' in i]

        # check if all the same sample rate
        if len(set([i[1] for i in video_audio_files])) > 1 or any([i[1] != 44100 for i in video_audio_files]):
            print(f'Video audio files for {data_dir} are not all the same sample rate. Resampling...')
            # resample
            video_audio_files = [librosa.resample(i[0], i[1], 44100) for i in video_audio_files if i[1] != 44100]

    audio_files.sort()
    processed_path = data_dir / 'processed'
    # make dir
    processed_path.mkdir(exist_ok=True)
    # isolate
    if use_video_audio:
        isolated_files = isolate_audio(audio_files, video_audio=video_audio_files, save_files=True, output_path=str(processed_path))
    else:
        isolated_files = isolate_audio(audio_files, save_files=True, output_path=str(processed_path))
    isolated_files.sort()
    # transcribe
    transcripts = []
    for file in isolated_files:
        transcript = transcribe(file)
        transcripts.append(transcript)
    # save transcripts
    for letter, transcript in zip(letter_assignments, transcripts):
        joblib.dump(transcript, processed_path / f'{letter}_transcript.pkl')

In [114]:
test_dir = Path('/media/landry/fastscratch/landry_dev_scratch/2023-10-06_000')

In [115]:
extract_transcripts(test_dir, use_video_audio=True)

Extracting audio from video files...

Moviepy - Running:
>>> "+ " ".join(cmd)
Moviepy - Command successful
Extracting audio from video files...

Moviepy - Running:
>>> "+ " ".join(cmd)
Moviepy - Command successful


TypeError: isolate_audio() got an unexpected keyword argument 'video_audio'

In [34]:
v_audios = get_video_audio_files(test_dir)

Extracting audio from video files...

Moviepy - Running:
>>> "+ " ".join(cmd)
Moviepy - Command successful
Extracting audio from video files...

Moviepy - Running:
>>> "+ " ".join(cmd)
Moviepy - Command successful


In [51]:
# make an additive mix of the video audio files
vmix = additive_mix([i[0] for i in v_audios])

In [73]:
# load in all the trimmed audio files
mic_audios = [librosa.load(i, sr=None) for i in test_dir.glob('derivatives/*_trimmed.wav')]
mic_audio_srs = [i[1] for i in mic_audios]
mic_audios = [i[0] for i in mic_audios]

In [65]:
vmix.shape

(88716500,)

In [75]:
mic_audios[0].shape

(88716096,)

In [77]:
# pad mic audio to match vmix
mic_audios = [np.pad(i, (0, vmix.shape[0]-len(i)), 'constant') for i in mic_audios]

In [83]:
rms_vmix = window_rms(vmix)
rms_mic = [window_rms(i) for i in mic_audios]